# CLAS12 RGA — Performance / Plotting

Post-hoc evaluation plots for the **clas12_RGA** experiment, built on the shared
`parnassus_core.performance` machinery (`Performance` + `PlotHelper`) bound to the
CLAS12-specific `Clas12RGASample` / `Clas12RGABinsConfig` in
`experiments/clas12_RGA/performance/`.

## Prerequisite: run the evaluation chain

`eval.py` writes a **raw** eval file (`*_eval_<k>k.root`) with structured `pflow`/`truth`
branches. The plots here read the **postprocessed** file, with `fastsim`, `pflow` and `truth`
collections joined by `eventNumber`, so run the full chain first (`scatE_filter` → `eval` →
`postprocess_eval`, see the "Evaluation" section of the experiment README) or use the
postprocessed file from the data release.


In [ ]:
from parnassus_core.experiments.clas12_RGA.performance.performance import (
    Clas12RGAPerformance,
    Clas12RGAPlotHelper,
    Clas12RGASample,
    Clas12RGAPerformanceConfig,
)

# Optional: nicer LaTeX labels if a TeX install is on PATH
from parnassus_core.experiments.clas12_RGA.performance.config import init_latex

init_latex()

## Configuration

`EVAL_FILE_CLASDIS` must point at the **postprocessed** `*Eval.root`. It is looked up in
`EVAL_DIR`, which defaults to `clasdis_v8_1_filtered/` in the repository root (the `eval_path`
of `evaluation/configs/v8/config_v8_1.yaml`); set `PARNASSUS_EVAL_DIR` to read it from
elsewhere. Add more entries under `samples` (and matching `Clas12RGASample` members) to overlay
additional datasets or fastsim variants.

In [ ]:
import os
from pathlib import Path

# Postprocessed v8.1 evaluation output. By default the eval_path of
# evaluation/configs/v8/config_v8_1.yaml, relative to the repository root; set
# PARNASSUS_EVAL_DIR to read it from elsewhere (e.g. the unpacked data release).
REPO_ROOT = Path("../../../../..")  # notebooks/ -> repository root
EVAL_DIR = Path(
    os.environ.get("PARNASSUS_EVAL_DIR", REPO_ROOT / "clasdis_v8_1_filtered")
)
# Figures are written here when save_fig is True (set PARNASSUS_FIG_DIR to change it).
FIG_DIR = Path(os.environ.get("PARNASSUS_FIG_DIR", REPO_ROOT / "plots" / "paper_figs"))

EVAL_FILE_CLASDIS = str(
    EVAL_DIR / "postprocess_dr_0_3_v8_filter_29M_6000k_test_eta49Eval.root"
)


config_dict = {
    "samples": {
        "clasdis": [
            {
                "name": "clasdis",
                "path": EVAL_FILE_CLASDIS,
                "pflow_type": "fm",
                "label": "Parnassus (FM)",
            },
        ],
    }
}

config = Clas12RGAPerformanceConfig.from_dict(config_dict)
config.sample_dict.keys()

## Load & process

`load()` reads the ROOT file(s); `process()` filters to common events, computes event
aggregates, matches objects, and derives residuals. Both are cached (safe to re-run).

In [ ]:
perf = Clas12RGAPerformance(config)
perf.load()
perf.process()

### Inspect available variables (optional)

Handy when adding custom metrics — see exactly which keys each collection exposes.

In [ ]:
sample_data = perf.sample_data_dict[Clas12RGASample.CLASDIS]
for name, data in sample_data.data_dict.items():
    print(f"--- {name} ({data.source.name}) ---")
    data.print_vars()

## Plot styling

`style_dict` is keyed by the `name` of each `Data` source: `PF` (full-sim reference),
`TR` (truth), and your fastsim entry name (`clasdis_v0`). `plot_truth=False` hides the
truth overlay in distribution plots.

In [ ]:
import numpy as np

style_dict = {
    "PF": {"linewidth": 2, "color": "#1f77b4", "alpha": 0.3},
    "TR": {"linewidth": 2, "color": "black", "alpha": 1},
    "clasdis": {"linewidth": 2, "color": "crimson"},
}

plot_helper = Clas12RGAPlotHelper(
    perf,
    plot_truth=False,
    style_dict=style_dict,
    font_size=25,
    show_chi2=False,  # False -> no chi2/ndf label on the panels
    # Union-join files pad each side: rows the reference does not cover have an
    # empty pflow collection and vice versa. Those rows land at npart == 0 and,
    # because np.sum over an empty array is 0.0, at Pxh = Pyh = ht_p = 0 -- a
    # fake spike sitting on the mode of the hadronic-momentum distribution. This
    # drops each collection's own empty events from the event_data grids.
    # Per-collection, so it does NOT reduce to the intersect-join comparison.
    # No effect on particle, DIS or pdgId plots, or on intersect-join files.
    drop_empty_events=True,
    # Panel columns of the event / DIS / particle grids; the variables wrap onto
    # extra rows (2 -> 2x2, 2x2, 4x2 for the two-column paper). None: one row of
    # panels. Per-figure override: plot_particle_features(..., ncols=3).
    ncols=2,
)

# Older samples lack some of the v8.1 particle variables (pre-v6: beta; v6-v7: vx/vy).
# Give them empty columns so those panels are left blank (and skipped for chi2)
# instead of raising a KeyError.
for sample_data in perf.sample_data_dict.values():
    for data in sample_data.data_dict.values():
        for var in plot_helper.particle_vars:
            for key in (var, f"{var}_res"):
                data.particle_data_all.setdefault(key, np.empty(0))

save_fig = True
if save_fig:
    FIG_DIR.mkdir(parents=True, exist_ok=True)

## Event features

`npart`, `Pxh`, `Pyh`, `ht_p` (and their residuals vs. truth), the v8.1 event-model
targets. `Pxh`/`Pyh` ($H_x$, $H_y$) are the vector px/py sums over every non-electron
particle (`pdgId != 11`); `ht_p` ($\sum p$) is the scalar sum of $|p|$. All event
residuals are absolute differences (reco − truth). Bin edges come from
`Clas12RGABinsConfig` — tune them in `performance/bins.py` or pass a `bins=` dict.

In [ ]:
fig = plot_helper.plot_event_features(residuals=False)
if save_fig:
    fig.savefig(f"{FIG_DIR}/results_event.pdf")

In [ ]:
fig = plot_helper.plot_event_features(residuals=True)
if save_fig:
    fig.savefig(f"{FIG_DIR}/results_event_residual.pdf")

## DIS features

`Q2`, `xB`, `W`, `y`, `nu` — reconstructed per event via the electron method in
`postprocess_eval.py` (`compute_dis: true`) and written as flat event branches. They are
loaded straight into `event_data`; bin edges come from `Clas12RGABinsConfig.dis_bins`
(tune in `performance/bins.py` or pass a `bins=` dict). Requires the postprocessed file
to have been produced with DIS enabled. These are the only relative residuals,
(reco − truth) / truth, labelled "Rel. Residual".

In [ ]:
fig = plot_helper.plot_dis_features(residuals=False)
if save_fig:
    fig.savefig(f"{FIG_DIR}/results_dis.pdf")

In [ ]:
fig = plot_helper.plot_dis_features(residuals=True)
if save_fig:
    fig.savefig(f"{FIG_DIR}/results_dis_residual.pdf")

## Particle features

`p`, `theta`, `phi`, `beta`, `vx`, `vy`, `vz` (and residuals; all absolute, reco − truth,
with `phi` wrapped into [−π, π]). Use `particle_mask_type` to restrict to a class
(`all` / `had` / `el` / `mu` / `charged` / `neutral`) and `log_scale_vars` for log-y (a
list of variable names, or `"all"`).

In [ ]:
fig = plot_helper.plot_particle_features(log_scale_vars=["beta", "vx", "vy", "vz"])
if save_fig:
    fig.savefig(f"{FIG_DIR}/results_particle.pdf")

In [ ]:
fig = plot_helper.plot_particle_features(residuals=True, log_scale_vars="all")
if save_fig:
    fig.savefig(f"{FIG_DIR}/results_particle_residual.pdf")

## Particle classification

Per-sample confusion matrix (truth vs. reconstructed class) plus the overall class
fraction. Requires that postprocessing was run with matching (`-m`).

In [ ]:
fig = plot_helper.plot_pdgId()
if save_fig:
    fig.savefig(f"{FIG_DIR}/results_pdgId.pdf")